# CA-SAM2: Prompt-Aligned 1024x1024 Distillation Pipeline
**Budget:** zsh (Free via Google Colab / Google One AI Pro & Zenodo)

This notebook trains and evaluates **MobileSAM (10.1M)** distilled from a **SAM2.1 Hiera-Tiny Teacher (39M)** at native **1024x1024** resolution on the ARCADE coronary angiogram dataset.

### Key Improvements:
1. **Resolution Parity:** Teacher and student both run at native 1024x1024.
2. **Prompt Alignment:** Dynamic forward pass eliminates the prompt-mismatch bug where student was penalized against a different point.
3. **Feature Alignment:** Both image embeddings are 64x64 (no lossy downsampling).
4. **Continuous Upsampling:** Evaluation uses smooth bilinear logit interpolation before thresholding.

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Dependencies
!pip install -q -U pip setuptools wheel
!pip install -q scikit-image scipy opencv-python-headless huggingface_hub timm pillow numpy torchvision

# Clone and install Meta SAM2
!git clone https://github.com/facebookresearch/sam2.git /content/sam2_meta
!pip install -q -e /content/sam2_meta --no-build-isolation

# Clone and install MobileSAM
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git

# Clone experiment repository
!git clone --branch fluoroscopy-sam2-1024 https://github.com/elakiyasivakumar/SAM2-Coronary-Angiography-VA.git /content/SAM2

In [ ]:
# 3. Download Free ARCADE Dataset from Zenodo (~451 MB)
import os, json, cv2, subprocess, numpy as np
from PIL import Image

DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)
ZIP_PATH = f"{DATA_ROOT}/arcade.zip"
EXTRACT_DIR = f"{DATA_ROOT}/arcade"

if not os.path.exists(ZIP_PATH):
    print("Downloading ARCADE dataset from Zenodo (100% Free)...")
    subprocess.run(["wget", "-q", "--show-progress",
                    "https://zenodo.org/records/10390295/files/arcade.zip?download=1",
                    "-O", ZIP_PATH], check=True)
    print("Extracting ARCADE archive...")
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", EXTRACT_DIR], check=True)

# Convert COCO polygon annotations to binary masks
def convert_split(img_dir, ann_file, out_img_dir, out_mask_dir):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)
    with open(ann_file) as f:
        coco = json.load(f)
    img_map = {im["id"]: im for im in coco["images"]}
    anns_by_img = {}
    for ann in coco["annotations"]:
        anns_by_img.setdefault(ann["image_id"], []).append(ann)
    for img_id, im_info in img_map.items():
        src_im = os.path.join(img_dir, im_info["file_name"])
        if not os.path.exists(src_im): continue
        dest_im = os.path.join(out_img_dir, im_info["file_name"])
        Image.open(src_im).save(dest_im)
        mask = np.zeros((im_info["height"], im_info["width"]), dtype=np.uint8)
        for ann in anns_by_img.get(img_id, []):
            for poly in ann.get("segmentation", []):
                pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts], 255)
        dest_msk = os.path.join(out_mask_dir, os.path.splitext(im_info["file_name"])[0] + ".png")
        Image.fromarray(mask).save(dest_msk)

base = f"{EXTRACT_DIR}/syntax"
convert_split(f"{base}/train/images", f"{base}/train/annotations/train.json",
              f"{DATA_ROOT}/arcade_train/images", f"{DATA_ROOT}/arcade_train/masks")
convert_split(f"{base}/val/images", f"{base}/val/annotations/val.json",
              f"{DATA_ROOT}/arcade_test/images", f"{DATA_ROOT}/arcade_test/masks")
print(f"Dataset Ready: Train images = {len(os.listdir(f"{DATA_ROOT}/arcade_train/images"))}, Test images = {len(os.listdir(f"{DATA_ROOT}/arcade_test/images"))}")

In [ ]:
# 4. Download Checkpoints from HuggingFace & MobileSAM (zsh Free)
from huggingface_hub import hf_hub_download
import urllib.request

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# MobileSAM base weights
mob_path = f"{CKPT_DIR}/mobile_sam.pt"
if not os.path.exists(mob_path):
    print("Downloading MobileSAM weights...")
    urllib.request.urlretrieve(
        "https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt", mob_path)

# Meta SAM2.1 base weights
base_t_path = f"{CKPT_DIR}/sam2.1_hiera_tiny.pt"
if not os.path.exists(base_t_path):
    print("Downloading Meta SAM2.1 base weights...")
    hf_hub_download(repo_id="facebook/sam2.1-hiera-tiny", filename="sam2.1_hiera_tiny.pt", local_dir=CKPT_DIR)

# Fine-tuned 1024 Fluoroscopy Teacher Checkpoint
teacher_path = f"{CKPT_DIR}/sam2_fluoroscopy_1024.pt"
if not os.path.exists(teacher_path):
    print("Downloading fine-tuned 1024 Teacher checkpoint...")
    hf_hub_download(repo_id="Elakiya17/fluoroscopy-sam2", filename="sam2_fluoroscopy_1024.pt", local_dir=CKPT_DIR)

print("All checkpoints ready:", os.listdir(CKPT_DIR))

In [ ]:
# 5. Set Environment Variables for 1024 Distillation
os.environ["DATA_DIR"] = "/content/data/arcade_train"
os.environ["TEST_DIR"] = "/content/data/arcade_test"
os.environ["MOBILE_CKPT"] = "/content/checkpoints/mobile_sam.pt"
os.environ["TEACHER_BASE_CKPT"] = "/content/checkpoints/sam2.1_hiera_tiny.pt"
os.environ["TEACHER_CKPT"] = "/content/checkpoints/sam2_fluoroscopy_1024.pt"
os.environ["SAM2_REPO"] = "/content/sam2_meta"
os.environ["PYTHONPATH"] = "/content/sam2_meta:/content/SAM2"
print("Environment configured.")

In [ ]:
# 6. Run Training: Prompt-Aligned Response KD (Ablation 2)
# Ablations:
#   --ablation 1 : GT Only baseline
#   --ablation 2 : GT + Prompt-Aligned Response KD
#   --ablation 4 : GT + Prompt-Aligned Response KD + clDice

!python /content/SAM2/distill/distill_student_1024.py --ablation 2 --epochs 30 --batch_size 4

In [ ]:
# 7. Visualize Model Overlays Side-by-Side
import matplotlib.pyplot as plt
import glob, torch
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from mobile_sam import sam_model_registry

# Load student
student = sam_model_registry["vit_t"](checkpoint="/content/checkpoints/mobile_sam.pt").cuda().eval()
ckpt = "/home/jupyter/mobilesam_1024_abl2.pt"
if os.path.exists(ckpt):
    student.load_state_dict(torch.load(ckpt, map_location="cuda"))

# Pick first test sample
test_imgs = sorted(glob.glob("/content/data/arcade_test/images/*.png"))
test_msks = sorted(glob.glob("/content/data/arcade_test/masks/*.png"))
img_pil = Image.open(test_imgs[0]).convert("RGB")
msk_np = np.array(Image.open(test_msks[0]).convert("L"))
ys, xs = np.where(msk_np > 0)
cx, cy = int(xs.mean()), int(ys.mean())

norm_1024 = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
img_t = norm_1024(img_pil).unsqueeze(0).cuda()
pt = torch.tensor([[[cx / msk_np.shape[1] * 1024.0, cy / msk_np.shape[0] * 1024.0]]], dtype=torch.float32, device="cuda")
lbl = torch.ones(1, 1, dtype=torch.int, device="cuda")

with torch.no_grad():
    emb = student.image_encoder(img_t)
    sp, dp = student.prompt_encoder(points=(pt, lbl), boxes=None, masks=None)
    logits, _ = student.mask_decoder(
        image_embeddings=emb,
        image_pe=student.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sp, dense_prompt_embeddings=dp, multimask_output=False)
    logits_up = F.interpolate(logits, size=(msk_np.shape[0], msk_np.shape[1]), mode="bilinear", align_corners=False)
    pred_mask = (logits_up[0, 0].cpu().numpy() > 0.0).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_pil)
axes[0].scatter([cx], [cy], c="red", s=40, label="Prompt Click")
axes[0].set_title("Original Fluoroscopy + Prompt")
axes[0].legend()
axes[1].imshow(msk_np, cmap="gray")
axes[1].set_title("Ground Truth Vessel Mask")
axes[2].imshow(pred_mask, cmap="viridis")
axes[2].set_title("MobileSAM Prediction (1024 Distilled)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()